<a href="https://colab.research.google.com/github/mesata/Facial-Expression-Recognition/blob/main/model_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader, TensorDataset

In [2]:
device = torch.device('cuda')
print(f"გამოყენებული მოწყობილობა: {device}")

გამოყენებული მოწყობილობა: cuda


In [4]:
class Architecture4(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.4)
        )

        self.classifier = nn.Sequential(
            nn.Linear(128 * 6 * 6, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 7)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

In [5]:
import wandb

run = wandb.init(project="facial-expression-recognition", job_type="inference")

artifact = run.use_artifact('best_emotion_model:v0', type='model')
artifact_dir = artifact.download()
model = Architecture4().to(device)
model.load_state_dict(torch.load(f"{artifact_dir}/best_model.pth", map_location=device))
model.eval()

run.finish()

wandb:   1 of 1 files downloaded.  


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
X_test_np = np.load('/content/drive/MyDrive/fer2013_processed/X_test.npy')
y_test_np = np.load('/content/drive/MyDrive/fer2013_processed/y_test.npy')

if len(X_test_np.shape) == 3:
    X_test_np = X_test_np.reshape(-1, 1, 48, 48)
elif X_test_np.shape[1] != 1 and len(X_test_np.shape) == 4:
    X_test_np = X_test_np.reshape(-1, 1, 48, 48)

X_test = torch.tensor(X_test_np, dtype=torch.float32)

if X_test.max() > 1.0:
    X_test = X_test / 255.0

y_test = torch.tensor(y_test_np, dtype=torch.long)

test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

final_accuracy = (correct / total) * 100
print("\n final result")
print(f"total tested: {total}")
print(f"final accuracy on test set: {final_accuracy:.2f}%")


 final result
total tested: 3589
final accuracy on test set: 48.01%


OVERFITTING o ara:(